# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² mass spectrometry dataset using the [`mlcroissant`](https://mlcommons.org/croissant) library. The workflow includes dataset metadata loading, record set overview, data extraction, exploratory data analysis (EDA), and simple visualization.

### Dataset Source
The dataset is referenced by its Croissant schema URL:

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the URL to the dataset Croissant schema
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

dataset = mlc.Dataset(croissant_url)

# Metadata can be accessed directly
print(f"Dataset Title: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"License: {dataset.metadata.license}\n")
print(f"Keywords: {dataset.metadata.keywords}\n")
print(f"Number of record sets: {len(dataset.metadata.record_sets)}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

- We list record sets, and for each record set, enumerate the available fields/columns and their `@id`s.

In [ ]:
from pprint import pprint

print("# Record Set Overview\n")
for rs in dataset.metadata.record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else 'No description'}")
    print(f"  Number of fields: {len(rs.fields)}")
    print(f"  Field IDs and Names:")
    for f in rs.fields:
        print(f"    - @id: {f.id}, name: {getattr(f, 'name', '-')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We'll extract the first record set and use the field `@id`s for demonstration.

In [ ]:
# List all record set IDs
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print(f"Available record set IDs: {record_set_ids}")

# Load data from all record sets into pandas DataFrames, indexed by record set @id
dataframes = {}
for rsid in record_set_ids:
    try:
        # Returns Python dicts (records)
        recs = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(recs)
        dataframes[rsid] = df
        print(f"Loaded records for RecordSet '{rsid}': {df.shape[0]} rows, {df.shape[1]} columns")
    except Exception as e:
        print(f"Could not load RecordSet {rsid}: {e}")

# Take the first record set as example for further exploration
example_record_set_id = record_set_ids[0]
print("\nColumns in the DataFrame for record set @id:", example_record_set_id)
print(dataframes[example_record_set_id].columns.tolist())
dataframes[example_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. The operations include removing outliers, transforming numeric distributions, or grouping data by key attributes for further analysis.

**Note:** All fields are referenced by their `@id` as required.

In [ ]:
# Choose a numeric field (`@id`) for demonstration.
# List fields for the first record set
example_record_set = None
for rs in dataset.metadata.record_sets:
    if rs.id == example_record_set_id:
        example_record_set = rs
        break
# Find first numeric field
numeric_field_id = None
for field in example_record_set.fields:
    if hasattr(field, 'data_type') and field.data_type in ['http://schema.org/Number', 'http://schema.org/Float', 'http://schema.org/Integer', 'http://mlcommons.org/croissant/Number', 'http://mlcommons.org/croissant/Float', 'http://mlcommons.org/croissant/Integer']:
        numeric_field_id = field.id
        break

if numeric_field_id is None:
    print("No numeric field detected for analysis.")
else:
    print(f"Using numeric field '@id': {numeric_field_id}\n")

    df = dataframes[example_record_set_id]

    # Drop missing values in this field
    df_clean = df.dropna(subset=[numeric_field_id])
    # Filter: example, values greater than the median
    threshold = df_clean[numeric_field_id].median()
    filtered_df = df_clean[df_clean[numeric_field_id] > threshold]

    print(f"Filtered records where {numeric_field_id} > median ({threshold}): {filtered_df.shape[0]} rows")
    print(filtered_df.head())

    # Normalization (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()

    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Find a candidate group-by field: the first textual/categorical field not the numeric one
    group_field_id = None
    for f in example_record_set.fields:
        if f.id != numeric_field_id and (hasattr(f, 'data_type') and f.data_type in ['http://schema.org/Text', 'http://mlcommons.org/croissant/Text']):
            group_field_id = f.id
            break

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id:
    plt.figure(figsize=(7, 4))
    filtered_df[numeric_field_id].hist(bins=30, edgecolor='k')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.title(f'Histogram of {numeric_field_id} (filtered)')
    plt.show()

    # If a group_field_id exists, do a boxplot
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10, 5))
        filtered_df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f'{numeric_field_id} grouped by {group_field_id}')
        plt.suptitle('')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, you have seen how to:
- Load a FAIR-compliant mass spectrometry dataset using `mlcroissant`.
- Explore available record sets and their field metadata (referenced by `@id`).
- Extract records and construct DataFrames for analysis.
- Apply basic filtering and normalization using field `@id`s.
- Visualize numeric field distributions and summary statistics.

This process can be extended to other datasets with Croissant schemas by following the same approach of using metadata and entity `@id` references, enabling reproducible and robust scientific data analysis.